# SOC Narrative: Small LLMs for UEBA / Insider Threat Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pancake2021/research_work_by_a_student/blob/main/notebooks/soc_narrative_demo.ipynb)

This notebook reproduces the core evaluation from the paper.
It loads a LoRA adapter from Hugging Face and evaluates it on the `dev_balanced_50` dataset.

**Requirements**: a GPU runtime (T4 or better) with ~12GB VRAM.

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate peft trl datasets scikit-learn pandas matplotlib sentencepiece

In [ ]:
import json
import sys
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore")

## 1. Config

Choose which checkpoint to evaluate:

In [ ]:
# Pick one:
CHECKPOINT = "Pankei/soc-narrative-grpo-strict128-qwen3-14b"
BASE_MODEL  = "Qwen/Qwen3-14B"
DATASET     = "Pankei/soc-narrative-dev-balanced-50"

print(f"Model: {CHECKPOINT}")
print(f"Base:  {BASE_MODEL}")
print(f"Data:  {DATASET}")

# Available checkpoints:
# Pankei/soc-narrative-sft-qwen3-14b           (SFT, Acc 0.74, Rec 0.88)
# Pankei/soc-narrative-grpo32-qwen3-14b        (GRPO-32, Acc 0.74, Rec 0.88)
# Pankei/soc-narrative-grpo-strict128-qwen3-14b (GRPO strict128, Acc 0.84, Rec 0.76)
# Pankei/soc-narrative-sft-qwen3.5-9b          (SFT, Acc 0.74, Rec 0.52)

## 2. Load Dataset

In [ ]:
dataset = load_dataset(DATASET, split="dev")
print(f"Loaded {len(dataset)} examples")
print(f"Labels: {pd.Series(dataset['risk_label']).value_counts().to_dict()}")

# Show one example
ex = dataset[0]
print(f"\nExample:")
print(f"  risk_label: {ex['risk_label']}")
print(f"  prompt (first 300 chars): {ex['prompt'][:300]}...")
print(f"  target: {ex['target']}")

## 3. Load Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA adapter
model = PeftModel.from_pretrained(model, CHECKPOINT)
model.eval()
print(f"Model loaded. Trainable params: {model.num_parameters(only_trainable=True):,}")

## 4. Inference

In [ ]:
import re

def parse_label(response_text):
    """Extract risk label from model response."""
    if not response_text:
        return None
    text = response_text.strip().lower()
    # Try "Риск:" or "risk:" marker first
    for marker in ("риск:", "risk:"):
        if marker in text:
            tail = text.split(marker, 1)[1]
            match = re.search(r'\b(normal|suspicious|malicious)\b', tail)
            if match:
                return match.group(1)
    # Fallback: find anywhere
    match = re.search(r'\b(normal|suspicious|malicious)\b', text)
    return match.group(1) if match else None


def generate(model, tokenizer, prompt, max_new=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


results = []
for i, ex in enumerate(dataset):
    response = generate(model, tokenizer, ex['prompt'])
    pred = parse_label(response)
    true = ex['risk_label']
    results.append({"true": true, "pred": pred, "response": response})
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(dataset)}]")

## 5. Results

In [ ]:
true_labels = [r["true"] for r in results]
pred_labels = [r["pred"] if r["pred"] else "unknown" for r in results]

print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(true_labels, pred_labels, labels=["normal", "malicious"], zero_division=0))

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(true_labels, pred_labels, labels=["normal", "malicious"])
print(f"           Pred normal  Pred malicious")
print(f"True normal  {cm[0][0]:>5}      {cm[0][1]:>5}")
print(f"True malicious {cm[1][0]:>4}      {cm[1][1]:>5}")

acc = accuracy_score(true_labels, pred_labels)
rec_mal = recall_score(true_labels, pred_labels, pos_label="malicious")
f1_macro = f1_score(true_labels, pred_labels, average="macro")

print(f"\nAccuracy:      {acc:.4f}")
print(f"Macro F1:      {f1_macro:.4f}")
print(f"Recall (mal):  {rec_mal:.4f}")

## 6. Sample Predictions

In [ ]:
print("=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)
for i, r in enumerate(results[:10]):
    correct = "✅" if r["true"] == r["pred"] else "❌"
    print(f"{i+1}. True: {r['true']:>10} | Pred: {str(r['pred']):>10} {correct}")
    print(f"   Response: {r['response'][:150]}")
    print()